# Word ↔ brain decoding with HTFA (Mitchell et al., 2008)

Mitchell et al. (2008, *Science* 320:1191) recorded fMRI while nine participants viewed 60 concrete nouns (12 categories of 5 words, e.g. *dog*, *hammer*, *house*), each with a line drawing, six times each. They showed that a linear map from word features to voxel activity predicts the images of nouns that were left out of the training set. This notebook repeats the two directions of that analysis with HTFA factor weights in place of voxels and modern sentence embeddings in place of the original verb co-occurrence features:

1. **brain prediction**: word embedding → HTFA weights → predicted image of the held-out word in the held-out participant's voxels;
2. **word decoding**: the held-out participant's real HTFA weights → predicted embedding → which of the 60 nouns?

Both mappings are learned on the other 8 participants and never see the held-out word in any participant.

TBD_RUNTIME

**Data.** The nine `data-science-P*.mat` files (about 60 MB each) come from the [dataset page](https://www.cs.cmu.edu/afs/cs/project/theo-73/www/science2008/data.html) and are cached in `~/.cache/htfa/mitchell2008` (or `$HTFA_DATA/mitchell2008`). In Colab, the next cell installs `htfa` from GitHub with the plotting extras, `hypertools` and `sentence-transformers`.

In [ ]:
import sys

if "google.colab" in sys.modules:  # Colab: install htfa and the extra packages
    %pip install -q "htfa[plot,nilearn] @ git+https://github.com/ContextLab/htfa"
    %pip install -q hypertools sentence-transformers

## 1. Download

Each file holds one participant: 360 trial images (60 nouns × 6 presentations). Per the dataset documentation, each trial image is the mean percent signal change 4 to 8 s after stimulus onset; the data were corrected for slice timing, motion and linear trend, high-pass filtered, normalized to MNI space with SPM and resampled to 3 × 3 × 6 mm voxels.

In [ ]:
import os
import pickle
import time
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import scipy.io
from joblib import Parallel, delayed
from nilearn import datasets, plotting
from sklearn.exceptions import ConvergenceWarning

import htfa
from htfa.tfa import standardize_data

t_notebook = time.perf_counter()
htfa.set_autoinstall(False)
print("htfa", htfa.__version__)
print("CPUs:", os.cpu_count(), "| load average (1, 5, 15 min):", os.getloadavg())

URL = "https://www.cs.cmu.edu/afs/cs/project/theo-73/www/science2008"
DATA_HOME = Path(os.environ.get("HTFA_DATA", Path.home() / ".cache" / "htfa"))
CACHE = DATA_HOME / "mitchell2008"
CACHE.mkdir(parents=True, exist_ok=True)


def fetch(name):
    """Download one file of the dataset into CACHE (once) and return its path."""
    path = CACHE / name
    if not path.exists():
        tmp = path.with_suffix(".part")
        urllib.request.urlretrieve(f"{URL}/{name}", tmp)
        tmp.rename(path)
    return path


t0 = time.perf_counter()
files = [fetch(f"data-science-P{p}.mat") for p in range(1, 10)]
print(
    f"{len(files)} files, {sum(f.stat().st_size for f in files) / 1e6:.0f} MB, "
    f"{time.perf_counter() - t0:.0f} s (0 s when already cached)"
)

## 2. Load the trials and the voxel coordinates

`meta.colToCoord` gives the (1-based) voxel indices of every column of `data`. The dataset's `functions.tar.gz` converts them to MNI millimetres with a fixed matrix (`get_mni_coordinates.m`), copied below; its in-plane step is 3.125 mm. Each participant has a different set of voxels, which HTFA allows: the factors are functions of MNI coordinates, not of voxel indices.

In [ ]:
# get_mni_coordinates.m from the dataset's functions.tar.gz: 1-based voxel
# indices (i, j, k) -> MNI millimetres
VOX_TO_MNI = np.array(
    [
        [-3.125, 0.0, 0.0, 81.25],
        [0.0, 3.125, 0.0, -115.625],
        [0.0, 0.0, 6.0, -54.0],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
AFFINE = VOX_TO_MNI.copy()  # the same map for 0-based indices, for NIfTI images
AFFINE[:3, 3] += VOX_TO_MNI[:3, :3].sum(axis=1)


def load_participant(path):
    m = scipy.io.loadmat(path)
    meta, info = m["meta"][0, 0], m["info"][0]
    X = np.vstack([trial[0].ravel() for trial in m["data"][:, 0]]).T  # (V, 360)
    ijk = meta["colToCoord"].astype(int) - 1  # 0-based voxel indices
    coords = nib.affines.apply_affine(AFFINE, ijk)
    words = np.array([str(t["word"][0]) for t in info])
    cats = np.array([str(t["cond"][0]) for t in info])
    shape = tuple(int(meta[d][0, 0]) for d in ("dimx", "dimy", "dimz"))
    return dict(X=X, coords=coords, ijk=ijk, words=words, cats=cats, shape=shape)


subjects = [load_participant(f) for f in files]
# vocabulary: the 60 nouns, sorted by category and then alphabetically
vocab_pairs = sorted(set(zip(subjects[0]["cats"], subjects[0]["words"], strict=True)))
vocab = np.array([w for _, w in vocab_pairs])
vocab_cats = np.array([c for c, _ in vocab_pairs])
for p, s in enumerate(subjects, 1):
    counts = {w: int((s["words"] == w).sum()) for w in vocab}
    assert set(s["words"]) == set(vocab) and set(counts.values()) == {6}, p
    assert all(
        c == dict(vocab_pairs)[w] for w, c in zip(s["words"], s["cats"], strict=True)
    )
print("voxels per participant:", [s["X"].shape[0] for s in subjects])
print("trials per participant:", {s["X"].shape[1] for s in subjects})
print("grid:", subjects[0]["shape"], "| categories:", len(set(vocab_cats)))
print("first 12 trials of P1:", list(subjects[0]["words"][:12]))

### Check: do the coordinates land in the brain?

TBD_COORDS

In [ ]:
def to_nifti(values, s):
    """Put a vector of voxel values of participant s into a 3D NIfTI image."""
    vol = np.zeros(subjects[s]["shape"])
    i, j, k = subjects[s]["ijk"].T
    vol[i, j, k] = values
    return nib.Nifti1Image(vol, AFFINE)


mni_mask = datasets.load_mni152_brain_mask()
inv = np.linalg.inv(mni_mask.affine)
mask_data = mni_mask.get_fdata() > 0
for p, s in enumerate(subjects, 1):
    idx = np.rint(nib.affines.apply_affine(inv, s["coords"])).astype(int)
    inside = mask_data[tuple(idx.T)].mean()
    lo, hi = s["coords"].min(0), s["coords"].max(0)
    print(
        f"P{p}: {inside:.1%} of voxels inside the MNI152 brain mask; "
        f"x {lo[0]:.0f}..{hi[0]:.0f}, y {lo[1]:.0f}..{hi[1]:.0f}, "
        f"z {lo[2]:.0f}..{hi[2]:.0f} mm"
    )
plotting.plot_glass_brain(
    to_nifti(np.ones(len(subjects[0]["coords"])), 0),
    display_mode="lyrz",
    cmap="Blues",
    vmin=0,
    vmax=1.5,
    title="P1 voxels",
)
plt.show()

## 3. Fit HTFA (K = 100)

The fit is unsupervised: it sees the 360 trial images of each participant, not the word labels. Each iteration solves for the centers and widths on a random subset of at most 10,000 voxels (`max_voxels=10000`; all 360 trials); the weights always use all voxels (see the *Performance* page). `init='ncorr'` uses one start per participant instead of the three of `init='auto'`. The fitted model is pickled to the cache directory, so later runs load it.

TBD_FIT

**Design choice.** This template is fitted with all nine participants, including the one held out below; the held-out participant's word labels are never used, but its unlabelled images shaped the template. Section 9 repeats the analysis with templates fitted without the held-out participant.

In [ ]:
fit_path = CACHE / "htfa_K100.pkl"
Xs = [s["X"] for s in subjects]
Rs = [s["coords"] for s in subjects]
fit_kwargs = dict(
    K=100, max_voxels=10000, init="ncorr", backend="numpy", n_jobs=3, random_state=0
)
if fit_path.exists():  # a pickle this notebook wrote itself; load only your own
    with open(fit_path, "rb") as f:
        cached = pickle.load(f)
    model, fit_seconds = cached["model"], cached["fit_seconds"]
    print(
        f"loaded {fit_path.name}; the fit took {fit_seconds:.0f} s at load "
        f"{np.round(cached['load'], 1)} (1, 5, 15 min) when it was computed"
    )
else:
    load_avg = os.getloadavg()
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model = htfa.fit(Xs, coords=Rs, **fit_kwargs).model
    fit_seconds = time.perf_counter() - t0
    with open(fit_path, "wb") as f:
        pickle.dump({"model": model, "fit_seconds": fit_seconds, "load": load_avg}, f)
    print(f"fit time: {fit_seconds:.0f} s (load average before: {np.round(load_avg)})")
    print("warnings:", sorted({w.category.__name__ for w in caught}))
res = htfa.HTFAResults(model, htfa.load(Xs, coords=Rs))
print("global iterations:", model.n_iter_, "| converged:", model.converged_)
print("weights per participant:", {w.shape for w in res.weights})
plotting.plot_connectome(
    np.zeros((100, 100)),
    res.global_centers,
    node_size=12,
    node_color="tab:red",
    display_mode="lyrz",
    title="HTFA template centers (K = 100)",
)
plt.show()

## 4. One weight vector per word

For each participant, the 6 presentations of each noun are averaged: the weights give a (60, K) matrix per participant, and the per-voxel z-scored data (the same z-scoring the fit applied) give a (60, V) matrix of word images. The last line measures how well K = 100 factors can represent a word image when the weights are known.

TBD_WORDS

In [ ]:
# Per-word means: 6 presentations -> one weight vector (K) and one image (V)
Ws, imgs = [], []
for s, w in zip(subjects, res.weights, strict=True):
    Z = standardize_data(s["X"], "auto")  # the per-voxel z-scoring the fit used
    Ws.append(np.stack([w[:, s["words"] == v].mean(axis=1) for v in vocab]))
    imgs.append(np.stack([Z[:, s["words"] == v].mean(axis=1) for v in vocab]))
Fs = [res.factors(s) for s in range(len(subjects))]
print("per-word weights:", Ws[0].shape, "| per-word images of P1:", imgs[0].shape)

# How much of the per-word images the K = 100 factors can represent at all
fit_r = [
    np.mean([np.corrcoef(F @ W[i], img[i])[0, 1] for i in range(60)])
    for F, W, img in zip(Fs, Ws, imgs, strict=True)
]
print("mean r(F w, word image) per participant:", np.round(fit_r, 2))

## 5. Word embeddings

**Model.** `Qwen/Qwen3-Embedding-0.6B` (Apache-2.0, 596M parameters, revision `97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3`). Its model card reports a mean task score of 64.33 on MTEB (Multilingual) and 70.70 on MTEB (English, v2), above the other models of 0.6B parameters or fewer that the card lists (e.g. multilingual-e5-large-instruct, 63.22 and 65.53); the card's comparison scores were taken from the MTEB leaderboard on 2025-05-24. It is not gated (it downloads without a Hugging Face token; the cell removes any token from the environment) and runs on a CPU or a free Colab runtime. EmbeddingGemma-300m, the first suggestion, is gated (its licence must be accepted on Hugging Face before download).

**hypertools.** `hypertools.tools.format_data` (hypertools 1.1) accepts a sentence-transformers model id as its `vectorizer`, so the embeddings below are computed through hypertools; `semantic=None` skips the topic-model stage and `ppca=False` leaves the 1024 dimensions as they are. Each noun is embedded on its own, without a prompt.

TBD_EMBED

In [ ]:
for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):  # the model is not gated
    os.environ.pop(key, None)
import hypertools as hyp  # noqa: E402

EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
emb_path = CACHE / "embeddings_Qwen3-Embedding-0.6B.npz"
if emb_path.exists():
    E = np.load(emb_path)["E"]
else:
    t0 = time.perf_counter()
    E = np.asarray(
        hyp.tools.format_data(
            list(vocab), vectorizer=EMBED_MODEL, semantic=None, corpus=None, ppca=False
        )[0]
    )
    np.savez(emb_path, E=E, vocab=vocab)
    print(f"embedding time: {time.perf_counter() - t0:.0f} s")
print("embeddings:", E.shape)
C = np.corrcoef(E)
same = np.equal.outer(vocab_cats, vocab_cats) & ~np.eye(60, dtype=bool)
print(
    f"mean r between embeddings: same category {C[same].mean():.3f}, "
    f"different categories {C[~np.equal.outer(vocab_cats, vocab_cats)].mean():.3f}"
)
hyp.plot(
    E,
    ".",
    hue=list(vocab_cats),
    labels=list(vocab),
    ndims=2,
    reduce="PCA",
    legend=True,
    backend="matplotlib",
    size=[9, 7],
    title="Qwen3-Embedding-0.6B word embeddings (PCA)",
)

## 6. Leave-one-participant-out, leave-one-word-out decoding

For each held-out participant *s* and each held-out word *i* (9 × 60 folds):

- **Normalization.** Each participant's (59, K) training weights are z-scored per factor across the 59 training words; the held-out participant's weights use the mean and SD of its own 59 training words.
- **Brain prediction.** Ridge regression from the embedding (1024) to the z-scored weights (K), trained on the 8 × 59 (participant, word) pairs. The predicted weights for word *i*, rescaled by participant *s*'s SDs, give the predicted image `F_s @ w` in participant *s*'s voxels. It is compared with the real image of every word after subtracting participant *s*'s mean image over the 59 training words (Mitchell et al. also removed the mean image). Reconstruction accuracy is the Pearson *r* with the real image of word *i*.
- **Word decoding.** Ridge regression from the z-scored weights to the embedding, trained on the same pairs, applied to participant *s*'s real weights for word *i*. The predicted embedding is correlated with the 60 noun embeddings (all centered on the 59 training words' mean).
- **Ridge penalty.** Chosen in every fold from 17 values (1e-3 to 1e5) by 5-fold cross-validation over the 59 training words (all 8 participants' rows of a word in the same inner fold).

**Scores** (from the 60 × 60 matrix `S[i, j]` = correlation of the prediction for word *i* with candidate *j*):

- *top-1*: the true candidate has the highest correlation (chance 1/60 ≈ 0.017);
- *rank accuracy*: fraction of the 59 wrong candidates that score below the true one (chance 0.5);
- *2-vs-2*: Mitchell's test over all 1770 word pairs, correct when `S[i, i] + S[j, j] > S[i, j] + S[j, i]` (chance 0.5). Mitchell et al. retrained without both words of each pair; here the two predictions come from the two leave-one-word-out folds.

In [ ]:
ALPHAS = np.logspace(-3, 5, 17)


def ridge_solution(A, B):
    """Center A and B and take the SVD of A: ridge for every alpha at once."""
    a_mu, b_mu = A.mean(0), B.mean(0)
    U, sv, Vt = np.linalg.svd(A - a_mu, full_matrices=False)
    return a_mu, b_mu, Vt, sv, U.T @ (B - b_mu)


def ridge_predict(sol, A_new, alpha):
    a_mu, b_mu, Vt, sv, UtB = sol
    return (A_new - a_mu) @ (Vt.T @ ((sv / (sv**2 + alpha))[:, None] * UtB)) + b_mu


def ridge_cv_predict(A, B, groups, A_new, alphas, n_folds=5):
    """Ridge A -> B, alpha chosen by 5-fold CV over groups (words); predict A_new."""
    rng = np.random.default_rng(0)
    folds = np.array_split(rng.permutation(np.unique(groups)), n_folds)
    err = np.zeros(len(alphas))
    for f in folds:
        test = np.isin(groups, f)
        sol = ridge_solution(A[~test], B[~test])
        err += [((ridge_predict(sol, A[test], a) - B[test]) ** 2).sum() for a in alphas]
    best = alphas[np.argmin(err)]
    return ridge_predict(ridge_solution(A, B), A_new, best), best


def corr_rows(P, Q):
    """Pearson r of each row of P with each row of Q."""
    P = P - P.mean(1, keepdims=True)
    Q = Q - Q.mean(1, keepdims=True)
    P = P / np.linalg.norm(P, axis=1, keepdims=True)
    Q = Q / np.linalg.norm(Q, axis=1, keepdims=True)
    return P @ Q.T


def decode_participant(s, E, Ws, Fs, imgs):
    """Leave participant s and, in turn, each word out of the mappings.

    Returns S_brain[i, j] = r(image predicted for word i, real image of word j),
    S_word[i, j] = r(embedding predicted from word i's weights, embedding of j),
    ceiling[i] = r(F_s w_s[i], real image of word i), all in participant s.
    """
    train = [t for t in range(len(Ws)) if t != s]
    n = len(E)
    S_brain, S_word, ceiling = np.zeros((n, n)), np.zeros((n, n)), np.zeros(n)
    for i in range(n):
        keep = np.arange(n) != i
        # z-score each participant's factor weights over its 59 training words
        Z = np.stack(
            [(Ws[t][keep] - Ws[t][keep].mean(0)) / Ws[t][keep].std(0) for t in train]
        )  # (8, 59, K)
        mu, sd = Ws[s][keep].mean(0), Ws[s][keep].std(0)
        z_s = (Ws[s][i] - mu) / sd
        dev = imgs[s] - imgs[s][keep].mean(0)  # images minus the 59-word mean
        # a. embedding -> weights -> image. The 8 participants share the inputs,
        # so ridge on the 8 x 59 rows equals ridge on the participant-mean
        # targets with alpha / 8.
        w_hat, _ = ridge_cv_predict(
            E[keep], Z.mean(0), np.arange(n - 1), E[i : i + 1], ALPHAS / len(train)
        )
        S_brain[i] = corr_rows((Fs[s] @ (w_hat[0] * sd))[None], dev)[0]
        ceiling[i] = corr_rows((Fs[s] @ (Ws[s][i] - mu))[None], dev[i : i + 1])[0, 0]
        # b. weights -> embedding -> word
        e_hat, _ = ridge_cv_predict(
            Z.reshape(-1, Z.shape[-1]),
            np.tile(E[keep], (len(train), 1)),
            np.tile(np.arange(n - 1), len(train)),
            z_s[None],
            ALPHAS,
        )
        e_mu = E[keep].mean(0)
        S_word[i] = corr_rows(e_hat - e_mu, E - e_mu)[0]
    return S_brain, S_word, ceiling


def scores(S):
    """Top-1 accuracy, normalized rank accuracy and 2-vs-2 accuracy."""
    n, d = len(S), np.diag(S)
    rank = np.mean([(np.delete(S[i], i) < d[i]).mean() for i in range(n)])
    a, b = np.triu_indices(n, 1)
    two = np.mean(d[a] + d[b] > S[a, b] + S[b, a])
    return np.mean(S.argmax(1) == np.arange(n)), rank, two

TBD_DECODE

In [ ]:
t0 = time.perf_counter()
out = Parallel(n_jobs=3)(
    delayed(decode_participant)(s, E, Ws, Fs, imgs) for s in range(len(subjects))
)
S_brain, S_word, ceiling = (np.stack(x) for x in zip(*out, strict=True))
print(f"decoding time: {time.perf_counter() - t0:.0f} s")
recon_r = np.array([np.diag(S) for S in S_brain])  # (9, 60)
brain = np.array([scores(S) for S in S_brain])  # columns: top-1, rank, 2-vs-2
word = np.array([scores(S) for S in S_word])
print(
    "participant | recon r | ceiling r | brain rank | brain 2v2 | "
    "word top-1 | word rank | word 2v2"
)
for p in range(len(subjects)):
    print(
        f"P{p + 1:<10} | {recon_r[p].mean():7.3f} | {ceiling[p].mean():9.3f} | "
        f"{brain[p, 1]:10.3f} | {brain[p, 2]:9.3f} | {word[p, 0]:10.3f} | "
        f"{word[p, 1]:9.3f} | {word[p, 2]:8.3f}"
    )
print(
    f"{'mean':<11} | {recon_r.mean():7.3f} | {ceiling.mean():9.3f} | "
    f"{brain[:, 1].mean():10.3f} | {brain[:, 2].mean():9.3f} | "
    f"{word[:, 0].mean():10.3f} | {word[:, 1].mean():9.3f} | "
    f"{word[:, 2].mean():8.3f}"
)
print(
    "chance      |         |           |      0.500 |     0.500 |      "
    f"{1 / 60:.3f} |     0.500 |    0.500"
)

## 7. Permutation test

Each permutation assigns the 60 embeddings to the 60 nouns at random and reruns the whole procedure (both mappings, all 540 folds, the inner choice of the penalty). The statistic is the mean over the 9 participants; *p* = (1 + number of permutations at least as large) / (1 + 100).

TBD_PERM

In [ ]:
N_PERM = 100
perm_path = CACHE / f"permutations_{N_PERM}.npz"


def permuted_means(seed):
    """Relabel the 60 embeddings at random and rerun the whole decoding."""
    E_perm = E[np.random.default_rng(seed).permutation(60)]
    res_p = [decode_participant(s, E_perm, Ws, Fs, imgs) for s in range(len(subjects))]
    b = np.array([scores(r[0]) for r in res_p]).mean(0)
    w = np.array([scores(r[1]) for r in res_p]).mean(0)
    recon = np.mean([np.diag(r[0]).mean() for r in res_p])
    return np.r_[recon, b, w]


if perm_path.exists():
    null = np.load(perm_path)["null"]
else:
    t0 = time.perf_counter()
    null = np.array(
        Parallel(n_jobs=4)(delayed(permuted_means)(1000 + k) for k in range(N_PERM))
    )
    np.savez(perm_path, null=null)
    print(f"permutation time: {time.perf_counter() - t0:.0f} s")
observed = np.r_[recon_r.mean(), brain.mean(0), word.mean(0)]
names = [
    "recon r",
    "brain top-1",
    "brain rank",
    "brain 2v2",
    "word top-1",
    "word rank",
    "word 2v2",
]
print(f"{'statistic (mean of 9)':<22} observed  null mean  null 95th pct  p")
for k, name in enumerate(names):
    p = (1 + np.sum(null[:, k] >= observed[k])) / (1 + len(null))
    print(
        f"{name:<22} {observed[k]:8.3f}  {null[:, k].mean():9.3f}  "
        f"{np.percentile(null[:, k], 95):13.3f}  {p:.3f}"
    )

## 8. Figures

Per participant: mean reconstruction *r* (bars), the ceiling of section 4 (black ticks) and the 95th percentile of the permutation null of the 9-participant mean (dashed); rank and 2-vs-2 accuracy in both directions, and word top-1 accuracy (red dots).

In [ ]:
labels = [f"P{p}" for p in range(1, 10)]
x = np.arange(9)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].bar(x, recon_r.mean(1), color="tab:blue", label="predicted vs real")
ax[0].plot(x, ceiling.mean(1), "k_", ms=14, mew=2, label="ceiling: F w (real w)")
ax[0].axhline(
    np.percentile(null[:, 0], 95), color="gray", ls="--", label="permutation 95th pct"
)
ax[0].set(ylabel="mean r over 60 words", title="image reconstruction")
panels = ((ax[1], brain, "brain: which image?"), (ax[2], word, "word: which noun?"))
for a, M, title in panels:
    a.bar(x - 0.2, M[:, 1], 0.4, label="rank accuracy")
    a.bar(x + 0.2, M[:, 2], 0.4, label="2-vs-2 accuracy")
    a.axhline(0.5, color="k", ls="--", lw=1, label="chance")
    a.set(ylim=(0.3, 1), title=title)
ax[2].scatter(x, word[:, 0], color="tab:red", zorder=3, label="top-1 (chance 1/60)")
for a in ax:
    a.set_xticks(x, labels)
    a.legend(fontsize=7, loc="upper left")
ax[0].set_ylim(0, max(ceiling.mean(1).max(), recon_r.mean(1).max()) * 1.4)
plt.tight_layout()
plt.show()

Ranks of the true item among the 60 candidates over all 540 held-out words; under chance they are uniform.

TBD_RANKS

In [ ]:
def true_ranks(S):
    """Rank of the true item among the 60 candidates (1 = best), per held-out word."""
    return np.array([1 + np.sum(np.delete(row, i) > row[i]) for i, row in enumerate(S)])


ranks_b = np.concatenate([true_ranks(S) for S in S_brain])
ranks_w = np.concatenate([true_ranks(S) for S in S_word])
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
bins = np.arange(0.5, 61.5, 3)
for a, r, title in (
    (ax[0], ranks_b, "brain: rank of the real image"),
    (ax[1], ranks_w, "word: rank of the real noun"),
):
    a.hist(r, bins=bins, color="tab:blue")
    a.axhline(len(r) / 20, color="k", ls="--", lw=1, label="uniform (chance)")
    a.set(xlabel="rank among 60 (1 = best)", title=title)
ax[0].set_ylabel("held-out words (9 x 60)")
ax[0].legend()
plt.tight_layout()
plt.show()
print(
    f"median rank: brain {np.median(ranks_b):.0f}, word {np.median(ranks_w):.0f} "
    "(chance 30.5)"
)
print(
    f"rank 1: brain {np.mean(ranks_b == 1):.3f}, word {np.mean(ranks_w == 1):.3f}; "
    f"rank <= 5: brain {np.mean(ranks_b <= 5):.3f}, word {np.mean(ranks_w <= 5):.3f}"
)

Real and predicted images of four nouns for P1 (the nouns were fixed before the results were computed). Both images are the deviation from P1's mean image over the 59 training words, z-scored across voxels and drawn on the same color scale (±3).

TBD_EXAMPLES

In [ ]:
s = 0  # P1
examples = ["hammer", "house", "dog", "car"]  # chosen before the results were seen
z = lambda v: (v - v.mean()) / v.std()  # noqa: E731
fig, axes = plt.subplots(len(examples), 2, figsize=(12, 2.4 * len(examples)))
for row, w in zip(axes, examples, strict=True):
    i = int(np.flatnonzero(vocab == w)[0])
    keep = np.arange(60) != i
    real = imgs[s][i] - imgs[s][keep].mean(0)
    # rerun the word-i fold of participant s to get its predicted image
    train = [t for t in range(9) if t != s]
    Z = np.stack(
        [(Ws[t][keep] - Ws[t][keep].mean(0)) / Ws[t][keep].std(0) for t in train]
    )
    w_hat, _ = ridge_cv_predict(
        E[keep], Z.mean(0), np.arange(59), E[i : i + 1], ALPHAS / 8
    )
    pred = Fs[s] @ (w_hat[0] * Ws[s][keep].std(0))
    r = np.corrcoef(pred, real)[0, 1]
    assert np.isclose(r, S_brain[s, i, i])
    for a, v, t in (
        (row[0], real, f"{w}: real (z)"),
        (row[1], pred, f"{w}: predicted (z), r = {r:.2f}"),
    ):
        plotting.plot_glass_brain(
            to_nifti(z(v), s),
            axes=a,
            display_mode="lyrz",
            plot_abs=False,
            cmap="RdBu_r",
            vmax=3,
            colorbar=True,
            title=t,
        )
plt.show()

## 9. Templates fitted without the held-out participant

For each held-out participant, HTFA is fitted to the other 8 (same settings). The held-out participant is then fitted from that template: an `HTFA` with one subject, `max_global_iter=1`, the template as both the start and the prior, so its factors are tied to the template's order without its data affecting the template that the mappings use. The decoding is then rerun with these weights and factor images.

TBD_STRICT

In [ ]:
strict_path = CACHE / "htfa_K100_leave_one_out.pkl"
if strict_path.exists():  # a pickle this notebook wrote itself
    with open(strict_path, "rb") as f:
        strict = pickle.load(f)
else:
    strict = {"models": [], "seconds": [], "load": os.getloadavg()}
    for s in range(len(subjects)):
        t0 = time.perf_counter()
        train = [t for t in range(len(subjects)) if t != s]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            m8 = htfa.fit(
                [Xs[t] for t in train], coords=[Rs[t] for t in train], **fit_kwargs
            ).model
            template = (m8.global_centers_, m8.global_widths_)
            # the held-out participant: one prior-penalized fit from the template
            m1 = htfa.HTFA(max_global_iter=1, max_local_iter=10, **fit_kwargs).fit(
                [Xs[s]], [Rs[s]], starts=[template], init_template=template
            )
        strict["models"].append((m8, m1))
        strict["seconds"].append(time.perf_counter() - t0)
        print(f"held out P{s + 1}: {strict['seconds'][-1]:.0f} s")
    with open(strict_path, "wb") as f:
        pickle.dump(strict, f)
print(
    f"total: {sum(strict['seconds']):.0f} s (load average at the start: "
    f"{np.round(strict['load'], 1)})"
)
dist = [
    np.median(np.linalg.norm(m1.centers_[0] - m8.global_centers_, axis=1))
    for m8, m1 in strict["models"]
]
print(
    "median distance (mm) of the held-out centers from the template they started at:",
    np.round(dist, 1),
)

strict_out = []
for s, (m8, m1) in enumerate(strict["models"]):
    train = [t for t in range(len(subjects)) if t != s]
    order = train + [s]  # participants in the order of m8's subjects, then s
    res8 = htfa.HTFAResults(
        m8, htfa.load([Xs[t] for t in train], coords=[Rs[t] for t in train])
    )
    res1 = htfa.HTFAResults(m1, htfa.load([Xs[s]], coords=[Rs[s]]))
    weights = list(res8.weights) + list(res1.weights)
    factors = [res8.factors(k) for k in range(len(train))] + [res1.factors(0)]
    Ws_s, Fs_s = [None] * len(subjects), [None] * len(subjects)
    for t, w, F in zip(order, weights, factors, strict=True):
        Ws_s[t] = np.stack(
            [w[:, subjects[t]["words"] == v].mean(axis=1) for v in vocab]
        )
        Fs_s[t] = F
    strict_out.append(decode_participant(s, E, Ws_s, Fs_s, imgs))
brain_strict = np.array([scores(o[0]) for o in strict_out])
word_strict = np.array([scores(o[1]) for o in strict_out])
recon_strict = np.array([np.diag(o[0]).mean() for o in strict_out])
print("                       template with P_s | template without P_s")
print(f"reconstruction r       {recon_r.mean():17.3f} | {recon_strict.mean():.3f}")
for k, name in ((1, "brain rank accuracy"), (2, "brain 2-vs-2")):
    print(f"{name:<22} {brain[:, k].mean():17.3f} | {brain_strict[:, k].mean():.3f}")
for k, name in ((0, "word top-1"), (1, "word rank accuracy"), (2, "word 2-vs-2")):
    print(f"{name:<22} {word[:, k].mean():17.3f} | {word_strict[:, k].mean():.3f}")

## Summary

TBD_SUMMARY

In [ ]:
print(
    f"notebook run time: {time.perf_counter() - t_notebook:.0f} s "
    f"(load average now: {np.round(os.getloadavg(), 1)})"
)